In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers,models,utils
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense,BatchNormalization,Input,Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau



In [7]:
import string

# All uppercase letters except 'J' and 'Z'
classes = [ch for ch in string.ascii_uppercase if ch not in ['J', 'Z']]
label_map = {ch: idx for idx, ch in enumerate(classes)}  # {'A': 0, 'B': 1, ..., 'Y': 23}


In [ ]:

def load_images_from_folder(folder_path, label_map):
    data = []
    labels = []

    for label in os.listdir(folder_path):
        if label not in label_map:
            continue
        label_folder = os.path.join(folder_path, label)
        for file in os.listdir(label_folder):
            img_path = os.path.join(label_folder, file)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img_resized = cv2.resize(img, (28, 28)) 
            data.append(img_resized)
            labels.append(label_map[label])
    
    return np.array(data), np.array(labels)

# Load custom datasets
X_good, y_good = load_images_from_folder("C:\\Users\\tanin\\OneDrive\\Desktop\\emerging_trends\\Sign Language\\clean_good_lighting", label_map)
X_bad, y_bad = load_images_from_folder("C:\\Users\\tanin\\OneDrive\\Desktop\\emerging_trends\\Sign Language\\clean_bad_lighting", label_map)
X_extra, y_extra =load_images_from_folder("C:\\Users\\tanin\\OneDrive\\Desktop\\emerging_trends\\Sign Language\\clean_extra", label_map)

# Combine your custom image dataset
X_custom = np.concatenate([X_good,X_bad])
y_custom = np.concatenate([y_good,y_bad])


In [ ]:
def load_sign_mnist_csv(path, label_map):
    df = pd.read_csv(path)
    
    original_labels = {i: ch for i, ch in enumerate(string.ascii_uppercase)}
    df['label'] = df['label'].map(original_labels)
    df = df[df['label'].isin(label_map)]
    df['label'] = df['label'].map(label_map)

    X = df.drop('label', axis=1).values.reshape(-1, 28, 28).astype('uint8')
    y = df['label'].values
    return X, y

X_train_mnist, y_train_mnist = load_sign_mnist_csv('sign_mnist_train.csv', label_map)
X_test_mnist, y_test_mnist = load_sign_mnist_csv('sign_mnist_test.csv', label_map)


In [56]:
# Combine MNIST and custom
X_train_all = np.concatenate([X_train_mnist, X_custom])
y_train_all = np.concatenate([y_train_mnist, y_custom])

X_test_all = np.concatenate([X_test_mnist, X_extra])  # Combining CSV test with extra data
y_test_all = np.concatenate([y_test_mnist, y_extra])

# Normalize and reshape
X_train_all = X_train_all / 255.0
X_test_all = X_test_all / 255.0

X_train_all = X_train_all.reshape(-1, 28, 28, 1)
X_test_all = X_test_all.reshape(-1, 28, 28, 1)

In [57]:
num_classes = 24
y_train_cat = to_categorical(y_train_all, num_classes)
y_test_cat = to_categorical(y_test_all, num_classes)

In [ ]:
train_datagen = ImageDataGenerator(
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,
    validation_split=0.2 
)

# Training generator
train_generator = train_datagen.flow(
    X_train_all, y_train_cat,
    batch_size=32,
    subset='training'
)

# Validation generator (from training set)
validation_generator = train_datagen.flow(
    X_train_all, y_train_cat,
    batch_size=32,
    subset='validation'
)

# Test generator
test_datagen = ImageDataGenerator()
test_generator = test_datagen.flow(
    X_test_all, y_test_cat,
    batch_size=32,
    shuffle=False
)


In [ ]:
input_layer = Input(shape=(28, 28, 1))

# Convolutional layer block 1
x = Conv2D(64, (3, 3), activation="relu", padding="same")(input_layer)
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)
x = Dropout(0.5)(x)

# Convolutional layer block 2
x = Conv2D(128, (3, 3), activation="relu", padding="same")(x)
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)
x = Dropout(0.5)(x)

# Convolutional layer block 3
x = Conv2D(256, (3, 3), activation="relu", padding="same")(x)
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)
x = Dropout(0.5)(x)

x = Flatten()(x)

x = Dense(128, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

output_layer = Dense(24, activation="softmax")(x)


In [91]:
model = Model(inputs=input_layer, outputs=output_layer)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [ ]:
early_stopping = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True) 
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=0.00001) 

In [93]:
history = model.fit(
    train_generator,
    epochs=15,
    validation_data=test_generator,
)

Epoch 1/15
817/817 ━━━━━━━━━━━━━━━━━━━━ 140s 163ms/step - accuracy: 0.1282 - loss: 3.1995 - val_accuracy: 0.2772 - val_loss: 2.5056
Epoch 2/15
817/817 ━━━━━━━━━━━━━━━━━━━━ 100s 122ms/step - accuracy: 0.5137 - loss: 1.5080 - val_accuracy: 0.5479 - val_loss: 1.9093
Epoch 3/15
817/817 ━━━━━━━━━━━━━━━━━━━━ 102s 125ms/step - accuracy: 0.6813 - loss: 0.9511 - val_accuracy: 0.6556 - val_loss: 1.4319
Epoch 4/15
817/817 ━━━━━━━━━━━━━━━━━━━━ 120s 147ms/step - accuracy: 0.7616 - loss: 0.7048 - val_accuracy: 0.7080 - val_loss: 1.3787
Epoch 5/15
817/817 ━━━━━━━━━━━━━━━━━━━━ 118s 145ms/step - accuracy: 0.8097 - loss: 0.5638 - val_accuracy: 0.7062 - val_loss: 1.4600
Epoch 6/15
817/817 ━━━━━━━━━━━━━━━━━━━━ 120s 146ms/step - accuracy: 0.8431 - loss: 0.4590 - val_accuracy: 0.7524 - val_loss: 1.3249
Epoch 7/15
817/817 ━━━━━━━━━━━━━━━━━━━━ 118s 144ms/step - accuracy: 0.8571 - loss: 0.4158 - val_accuracy: 0.7257 - val_loss: 1.4711
Epoch 8/15
817/817 ━━━━━━━━━━━━━━━━━━━━ 121s 149ms/step - accuracy: 0.8771 -

In [94]:
# Evaluate the model on the validation set
val_loss, val_accuracy = model.evaluate(X_test_all, y_test_cat)
print(f'Validation Accuracy: {val_accuracy * 100:.2f}%')


314/314 ━━━━━━━━━━━━━━━━━━━━ 11s 34ms/step - accuracy: 0.9388 - loss: 0.2906
Validation Accuracy: 80.74%


In [ ]:
# Save model
model.save('asl_sign_language_model1.keras')
